In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

os.environ["AWS_REGION"] = "us-east-2"

spark = SparkSession.builder \
    .appName("test-bronze") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("SparkSession criada com sucesso")

SparkSession criada com sucesso


In [16]:
# Lê a Bronze Layer

BUCKET = "clashr-account-data-lake"
BATTLES_PATH = f"s3a://{BUCKET}/raw/battles/8Q8UQPP8L/2026-06-03/"
PROFILE_PATH = f"s3a://{BUCKET}/raw/profile/8Q8UQPP8L/2026-06-03/"

print(f"Lendo batalhas de: {BATTLES_PATH}")
df_battles = spark.read.option("multiLine", "true").json(BATTLES_PATH)

print(f"Lendo perfis de: {PROFILE_PATH}")
df_profile = spark.read.option("multiLine", "true").json(PROFILE_PATH)

print(f"\nBatalhas - Total de registros: {df_battles.count()}")
print(f"Perfis - Total de registros: {df_profile.count()}")

Lendo batalhas de: s3a://clashr-account-data-lake/raw/battles/8Q8UQPP8L/2026-06-03/
Lendo perfis de: s3a://clashr-account-data-lake/raw/profile/8Q8UQPP8L/2026-06-03/

Batalhas - Total de registros: 30
Perfis - Total de registros: 1


In [17]:
# Valida os dados de batalhas

print("\n=== SCHEMA DE BATALHAS ===")
df_battles.printSchema()

print("\n=== AMOSTRA DE DADOS (5 linhas) ===")
df_battles.show(5, truncate=False)

print("\n=== ESTATÍSTICAS ===")
print(f"Nulos em player_tag: {df_battles.filter(F.col('player_tag').isNull()).count()}")
print(f"Nulos em battle_time: {df_battles.filter(F.col('battle_time').isNull()).count()}")
print(f"Nulos em result: {df_battles.filter(F.col('result').isNull()).count()}")

print("\n=== DISTRIBUIÇÃO DE RESULTADOS ===")
df_battles.groupBy("result").count().show()

print("\n=== TIPOS DE BATALHA ===")
df_battles.groupBy("battle_type").count().show()


=== SCHEMA DE BATALHAS ===
root
 |-- battle_time: string (nullable = true)
 |-- battle_type: string (nullable = true)
 |-- crowns_opponent: long (nullable = true)
 |-- crowns_player: long (nullable = true)
 |-- opponent_deck: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- opponent_tag: string (nullable = true)
 |-- player_deck: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- player_elixir_avg: double (nullable = true)
 |-- player_tag: string (nullable = true)
 |-- player_trophies: long (nullable = true)
 |-- result: string (nullable = true)


=== AMOSTRA DE DADOS (5 linhas) ===
+--------------------+-----------+---------------+-------------+-------------------------------------------------------------------------------------------------------+------------+----------------------------------------------------------------------------------------------------+-----------------+----------+---------------+------+
|battle_time        

In [18]:
# Valida os dados de perfil

print("\n=== SCHEMA DE PERFIL ===")
df_profile.printSchema()

print("\n=== AMOSTRA DE DADOS ===")
df_profile.select("tag", "name", "level", "trophies", "wins", "losses").show(3)

print("\n=== ESTATÍSTICAS DE PERFIL ===")
df_profile.select(
    F.round(F.avg("trophies"), 0).alias("avg_trophies"),
    F.max("trophies").alias("max_trophies"),
    F.min("trophies").alias("min_trophies"),
    F.round(F.avg(F.col("wins") / F.col("battle_count") * 100), 2).alias("avg_win_rate")
).show()

print("\n=== COM CLAN vs SEM CLAN ===")
df_profile.withColumn(
    "tem_clan",
    F.when(F.col("clan_name").isNotNull(), "Com Clan").otherwise("Sem Clan")
).groupBy("tem_clan").count().show()

print("\n✅ Bronze Layer validada com sucesso!")


=== SCHEMA DE PERFIL ===
root
 |-- battle_count: long (nullable = true)
 |-- best_trophies: long (nullable = true)
 |-- cards: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- elixir_cost: long (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- level: long (nullable = true)
 |    |    |-- max_level: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- rarity: string (nullable = true)
 |-- clan_name: string (nullable = true)
 |-- clan_tag: string (nullable = true)
 |-- level: long (nullable = true)
 |-- losses: long (nullable = true)
 |-- name: string (nullable = true)
 |-- tag: string (nullable = true)
 |-- trophies: long (nullable = true)
 |-- wins: long (nullable = true)


=== AMOSTRA DE DADOS ===


+---------+--------+-----+--------+----+------+
|      tag|    name|level|trophies|wins|losses|
+---------+--------+-----+--------+----+------+
|8Q8UQPP8L|MingaL04|   55|   11210|3126|  2789|
+---------+--------+-----+--------+----+------+


=== ESTATÍSTICAS DE PERFIL ===
+------------+------------+------------+------------+
|avg_trophies|max_trophies|min_trophies|avg_win_rate|
+------------+------------+------------+------------+
|     11210.0|       11210|       11210|       52.85|
+------------+------------+------------+------------+


=== COM CLAN vs SEM CLAN ===
+--------+-----+
|tem_clan|count|
+--------+-----+
|Com Clan|    1|
+--------+-----+


✅ Bronze Layer validada com sucesso!
